In [4]:
# Data Download
# Download data from Yahoo Finance

import yfinance as yf
from datetime import date


def download_data(symbol, period, interval):
    tickerData = yf.Ticker(symbol)
    tickerDf = tickerData.history(
        period=period,
        interval=interval,
    )
    tickerDf = tickerDf[tickerDf.High != tickerDf.Low]

    tickerDf.to_csv(f"../data/{symbol.lower()}-{interval}-{period}.csv")
    return tickerDf


tradingData = download_data('^IXIC', '1y', '1h')
tradingData

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2023-04-25 09:30:00-04:00,11967.986328,11990.456055,11922.806641,11926.114258,0,0.0,0.0
2023-04-25 10:30:00-04:00,11929.163086,11951.214844,11916.118164,11933.771484,583734000,0.0,0.0
2023-04-25 11:30:00-04:00,11930.925781,11930.925781,11892.834961,11894.381836,498663000,0.0,0.0
2023-04-25 12:30:00-04:00,11893.890625,11900.666992,11842.454102,11862.843750,434447000,0.0,0.0
2023-04-25 13:30:00-04:00,11862.315430,11864.176758,11822.761719,11843.119141,386835000,0.0,0.0
...,...,...,...,...,...,...,...
2024-04-24 11:30:00-04:00,15713.906250,15730.808594,15648.197266,15666.854492,443501000,0.0,0.0
2024-04-24 12:30:00-04:00,15666.922852,15714.307617,15635.220703,15712.334961,362863000,0.0,0.0
2024-04-24 13:30:00-04:00,15712.663086,15755.391602,15692.557617,15693.396484,338679000,0.0,0.0


In [6]:
# Read data
import pandas as pd

def read_data(symbol, period, interval):
    data = pd.read_csv(f'../data/{symbol}-{period}-{interval}.csv')
    data.Datetime = pd.to_datetime(data.Datetime, utc=True)
    data.set_index('Datetime', inplace=False)
    data.drop(['Dividends', 'Stock Splits', 'Volume'], axis=1, inplace=True)
    return data


tradingData = read_data('^ixic', '1y', '1h')
tradingData

,Datetime,Open,High,Low,Close
0,2023-04-20 13:30:00+00:00,12039.096680,12122.727539,12039.096680,12110.858398
1,2023-04-20 14:30:00+00:00,12112.754883,12130.212891,12077.960938,12094.939453
2,2023-04-20 15:30:00+00:00,12094.689453,12114.124023,12081.461914,12112.148438
3,2023-04-20 16:30:00+00:00,12112.006836,12155.889648,12109.030273,12151.598633
4,2023-04-20 17:30:00+00:00,12152.539062,12155.239258,12082.914062,12083.585938
...,...,...,...,...,...
1751,2024-04-19 15:30:00+00:00,15431.418945,15446.855469,15313.109375,15313.916992
1752,2024-04-19 16:30:00+00:00,15312.722656,15401.906250,15312.081055,15361.449219
1753,2024-04-19 17:30:00+00:00,15362.106445,15425.594727,15311.298828,15311.298828
1754,2024-04-19 18:30:00+00:00,15308.803711,15323.899414,15235.889648,15282.343750


In [8]:
# Set MACD
import pandas_ta as ta

def set_macd(data):
    macd = ta.macd(data.Close)
    data['MACD'] = macd.loc[:, 'MACD_12_26_9']
    data['MACD_signal'] = macd.loc[:, 'MACDs_12_26_9']
    data['MACD_histogram'] = macd.loc[:, 'MACDh_12_26_9']
    return data

set_macd(tradingData)
tradingData

,Datetime,Open,High,Low,Close,MACD,MACD_signal,MACD_histogram
0,2023-04-20 13:30:00+00:00,12039.096680,12122.727539,12039.096680,12110.858398,NaN,NaN,NaN
1,2023-04-20 14:30:00+00:00,12112.754883,12130.212891,12077.960938,12094.939453,NaN,NaN,NaN
2,2023-04-20 15:30:00+00:00,12094.689453,12114.124023,12081.461914,12112.148438,NaN,NaN,NaN
3,2023-04-20 16:30:00+00:00,12112.006836,12155.889648,12109.030273,12151.598633,NaN,NaN,NaN
4,2023-04-20 17:30:00+00:00,12152.539062,12155.239258,12082.914062,12083.585938,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1751,2024-04-19 15:30:00+00:00,15431.418945,15446.855469,15313.109375,15313.916992,-146.358802,-121.116970,-25.241832
1752,2024-04-19 16:30:00+00:00,15312.722656,15401.906250,15312.081055,15361.449219,-152.629630,-127.419502,-25.210128
1753,2024-04-19 17:30:00+00:00,15362.106445,15425.594727,15311.298828,15311.298828,-159.803904,-133.896383,-25.907521
1754,2024-04-19 18:30:00+00:00,15308.803711,15323.899414,15235.889648,15282.343750,-165.913453,-140.299797,-25.613656


In [12]:
# Set MACD Signal
from tqdm import tqdm
tqdm.pandas()

def macd_trade_signal(data, current):
    if (
        all(data.loc[current - 3:current - 2, 'MACD'] <
            data.loc[current - 3:current - 2, 'MACD_signal']) and
        all(data.loc[current - 1:current, 'MACD'] >
            data.loc[current - 1:current, 'MACD_signal'])
    ):
        return 1
    if (
        all(data.loc[current - 3:current - 2, 'MACD'] >
            data.loc[current - 3:current - 2, 'MACD_signal']) and
        all(data.loc[current - 1:current, 'MACD'] <
            data.loc[current - 1:current, 'MACD_signal'])
    ):
        return -1
    return 0

def set_macd_trade_signal(data):
    data['MACD_trade_signal'] = data.progress_apply(
        lambda r: macd_trade_signal(data, r.name),
        axis='columns'
    )
    return data

set_macd_trade_signal(tradingData)
tradingData

100%|██████████| 1756/1756 [00:00<00:00, 5315.80it/s]


,Datetime,Open,High,Low,Close,MACD,MACD_signal,MACD_histogram,MACD_trade_signal
0,2023-04-20 13:30:00+00:00,12039.096680,12122.727539,12039.096680,12110.858398,NaN,NaN,NaN,0
1,2023-04-20 14:30:00+00:00,12112.754883,12130.212891,12077.960938,12094.939453,NaN,NaN,NaN,0
2,2023-04-20 15:30:00+00:00,12094.689453,12114.124023,12081.461914,12112.148438,NaN,NaN,NaN,0
3,2023-04-20 16:30:00+00:00,12112.006836,12155.889648,12109.030273,12151.598633,NaN,NaN,NaN,0
4,2023-04-20 17:30:00+00:00,12152.539062,12155.239258,12082.914062,12083.585938,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...
1751,2024-04-19 15:30:00+00:00,15431.418945,15446.855469,15313.109375,15313.916992,-146.358802,-121.116970,-25.241832,0
1752,2024-04-19 16:30:00+00:00,15312.722656,15401.906250,15312.081055,15361.449219,-152.629630,-127.419502,-25.210128,0
1753,2024-04-19 17:30:00+00:00,15362.106445,15425.594727,15311.298828,15311.298828,-159.803904,-133.896383,-25.907521,0
1754,2024-04-19 18:30:00+00:00,15308.803711,15323.899414,15235.889648,15282.343750,-165.913453,-140.299797,-25.613656,0


In [14]:
# Set Total Trade Signal

def set_total_trade_signal(data, total_trade_signal):
    data['Total_trade_signal'] = data.progress_apply(
        lambda r: total_trade_signal(r),
        axis='columns'
    )
    return data

def total_trade_signal(row):
    if (row.MACD_trade_signal == 1):
        return 1
    elif (row.MACD_trade_signal == -1):
        return -1
    else:
        return 0

set_total_trade_signal(tradingData, total_trade_signal)
print(f'number of trades: {tradingData[tradingData.Total_trade_signal != 0].shape[0]}')

tradingData


100%|██████████| 1756/1756 [00:00<00:00, 75579.25it/s]

number of trades: 126


,Datetime,Open,High,Low,Close,MACD,MACD_signal,MACD_histogram,MACD_trade_signal,Total_trade_signal
0,2023-04-20 13:30:00+00:00,12039.096680,12122.727539,12039.096680,12110.858398,NaN,NaN,NaN,0,0
1,2023-04-20 14:30:00+00:00,12112.754883,12130.212891,12077.960938,12094.939453,NaN,NaN,NaN,0,0
2,2023-04-20 15:30:00+00:00,12094.689453,12114.124023,12081.461914,12112.148438,NaN,NaN,NaN,0,0
3,2023-04-20 16:30:00+00:00,12112.006836,12155.889648,12109.030273,12151.598633,NaN,NaN,NaN,0,0
4,2023-04-20 17:30:00+00:00,12152.539062,12155.239258,12082.914062,12083.585938,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...
1751,2024-04-19 15:30:00+00:00,15431.418945,15446.855469,15313.109375,15313.916992,-146.358802,-121.116970,-25.241832,0,0
1752,2024-04-19 16:30:00+00:00,15312.722656,15401.906250,15312.081055,15361.449219,-152.629630,-127.419502,-25.210128,0,0
1753,2024-04-19 17:30:00+00:00,15362.106445,15425.594727,15311.298828,15311.298828,-159.803904,-133.896383,-25.907521,0,0
1754,2024-04-19 18:30:00+00:00,15308.803711,15323.899414,15235.889648,15282.343750,-165.913453,-140.299797,-25.613656,0,0


In [15]:
# Verify the strategy by backtesting it
import seaborn as sns
import matplotlib.pyplot as plt
from backtesting import Strategy
from backtesting import Backtest

def set_atr(data):
    data['ATR'] = ta.atr(data.High, data.Low, data.Close, length=7)
    return data


class MacdStrategy(Strategy):
    mysize = 3
    slcoef = 1.1
    TPSLRatio = 1.5

    def init(self):
        super().init()
        self.signal1 = self.I(lambda: self.data.Total_trade_signal)

    def next(self):
        super().next()
        slatr = self.slcoef * self.data.ATR[-1]
        TPSLRatio = self.TPSLRatio

        if self.signal1 == 1 and len(self.trades) == 0:
            sl1 = self.data.Close[-1] - slatr
            tp1 = self.data.Close[-1] + slatr * TPSLRatio
            self.buy(sl=sl1, tp=tp1, size=self.mysize)

        elif self.signal1 == -1 and len(self.trades) == 0:
            sl1 = self.data.Close[-1] + slatr
            tp1 = self.data.Close[-1] - slatr*TPSLRatio
            self.sell(sl=sl1, tp=tp1, size=self.mysize)

def show_heatmap(heatmap):
    # Convert multiindex series to dataframe
    heatmap_dataFrame = heatmap.unstack()
    plt.figure(figsize=(10, 8))
    sns.heatmap(heatmap_dataFrame, annot=True, cmap='viridis', fmt='.0f')
    plt.show()


def backtest_trading(data, cash):
    backtest = Backtest(data, MacdStrategy, cash=cash,
                        margin=1/30, commission=0.00)
    stats, heatmap = backtest.optimize(slcoef=[i/10 for i in range(10, 26)],
                                       TPSLRatio=[i/10 for i in range(10, 26)],
                                       # rsi_length=[5, 8, 10, 12, 14, 16],
                                       maximize='Return [%]', max_tries=300,
                                       random_state=0,
                                       return_heatmap=True)

    return stats, heatmap

set_atr(tradingData)
stats, heatmap = backtest_trading(tradingData, 25000)

print(stats)
show_heatmap(heatmap)



/var/folders/rk/xympl3kj42nf5b5zvnt_lrxh0000gn/T/ipykernel_25869/1095126640.py:40: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  backtest = Backtest(data, MacdStrategy, cash=cash,


AttributeError: Column 'ATR' not in data